In [ ]:
from langchain_ollama import ChatOllama
model = ChatOllama(model="gpt-oss:120b-cloud")
print('LLM is ready')

In [ ]:
# STEP 1: Load external knowledge base and convert into pages
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(file_path="C:\\Anand\\old_laptop_backup\\D drive data\\Anand\\material\\Training\\Gen_AI\\L2_Applied_Gen_AI\\external_knowledgebase_for_rag\\Python_Programming.pdf")
docs = loader.load()

#print(docs[0].page_content[:50])  # Print first 50 characters of the first page])
print('PDF file is loaded and converted into pages')

In [ ]:
# STEP 2: Split the external knowledge base i.e. create chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300, 
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)
split_docs = text_splitter.split_documents(docs)
print(f'Number of chunks created: {len(split_docs)}')

In [ ]:
# STEP 3: Create embeddings & vector stores using FAISS/Chroma etc.
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OllamaEmbeddings(model="nomic-embed-text")
faiss_vector_store = FAISS.from_documents(split_docs, embeddings)
print(faiss_vector_store)

In [ ]:
# STEP 4: Integrate with LLM & build retrieval chain
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.chains import RetrievalQA

user_prompt_template = """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
-----------------
{context}
Question: {question}
"""
custom_prompt = PromptTemplate(
    template=user_prompt_template, input_variables=["context", "question"]
)
parser = StrOutputParser()
model = ChatOllama(model="gpt-oss:120b-cloud")
retrieval_chain = RetrievalQA.from_chain_type(
    llm=model,
    chain_type="stuff", # Map_reduce, refine, map_rerank, stuff
    retriever=faiss_vector_store.as_retriever(search_kwargs={"k": 6}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": custom_prompt}

)

In [ ]:
# STEP 5: Build UI user interface
import gradio as gr

def chatbot_response(message, history):
    # Use the created RetrievalQA chain to get the answer
    response = retrieval_chain({"query": message})
    answer = response["result"]
    source_documents = response["source_documents"]

    # Format the response to include the answer and source documents (optional)
    formatted_response = f"{answer}" # You can add source documents here if desired

    return formatted_response

# Create the Gradio interface
iface = gr.ChatInterface(
    fn=chatbot_response,
    title="RAG Chatbot",
    description="Ask questions about Python programming based on the provided pdf."
)

# Launch the interface
iface.launch(share=True)